# GFlowNet vs PPO/REINFORCE vs MCMC — diversity on a multi-modal reward

**A tiny, CPU-only, from-scratch teaching notebook.**

> *Audience: an ML researcher working on GFlowNets / FlowGRPO for molecular design.*
> No `gym`, no external RL libs — just `numpy`, `torch`, `matplotlib`.

## The one idea

Three families of methods, three different stationary behaviours on the **same**
non-negative reward landscape $R(x)$ with several separated high-reward modes:

| Method | What it optimizes | Stationary behaviour | Diversity |
|---|---|---|---|
| **GFlowNet** (trajectory balance) | samples $x$ with prob. $\propto R(x)$ | **matches** the reward landscape | **high** — finds *all* the modes |
| **REINFORCE / PPO** | $\mathbb{E}[R(x)]$ (argmax-seeking) | collapses onto the **single best** cell | **low** — one mode |
| **MCMC** (Metropolis) | target $\propto R(x)$ | correct in the limit, but **mixes slowly** between separated modes | medium / slow |

**Why this matters for hit discovery.** In binder / molecule design we rarely want
*the* single highest-scoring candidate — the scorer is a noisy proxy, synthesis
fails, and we want a *batch* of structurally distinct hits to de-risk downstream.
Sampling $\propto R$ gives you a **diverse set of high-reward candidates** in one
shot. Reward-maximizing RL gives you one peak (and tends to exploit scorer
artifacts). This is the core argument for GFlowNets in generative design.

This is exactly the *search + learning* tension in my notes:
**[GRPO](https://app.notion.com/p/37f95008d76681499281fcd797cb3d1d)** (reward-maximizing
policy-gradient) vs **[Search + Learning for Molecular Design](https://app.notion.com/p/37f95008d7668176886cf77d703358bc)**
(GFlowNet as the *diversity-first* rival to MCTS / RL for synthesis-tree generation).

### The toy environment: a **hypergrid**

An $H\times H$ grid. An episode starts at the corner $(0,0)$ and at each step takes
one of three actions:

- `+x`  : move one cell right (if not at the right edge),
- `+y`  : move one cell up (if not at the top edge),
- `stop`: terminate here; the current cell $x$ is the generated object.

The terminal cell gets reward $R(x) > 0$ — a small positive background plus a few
sharp Gaussian-ish bumps near separated cells (the "modes"). The **target
distribution** we want to sample is $p^\*(x) = R(x) / \sum_{x'} R(x')$.

A hypergrid is the canonical GFlowNet sanity-check (Bengio et al., 2021): the state
space is a DAG (every cell has multiple parents -> multiple trajectories reach it),
which is precisely what makes trajectory balance non-trivial and what RL's
argmax behaviour ignores.


## 1. Setup, config & seeds

Everything is sized to run in **a couple of minutes on a CPU**. Knobs live in one
place so you can crank them up on Colab if you like.

In [ ]:
import time, math, random
import numpy as np
import matplotlib.pyplot as plt

# torch is optional-at-import-time so the notebook degrades gracefully, but for
# the real comparison we need it. On Colab it's preinstalled.
try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    HAVE_TORCH = True
except Exception as e:  # pragma: no cover
    HAVE_TORCH = False
    print("torch not importable:", e)

print("torch available:", HAVE_TORCH)
if HAVE_TORCH:
    torch.set_num_threads(max(1, os_cpu := __import__('os').cpu_count() or 1))
    print("torch", torch.__version__, "| threads:", os_cpu)


In [ ]:
# ----------------------------- CONFIG -------------------------------------
# Keep H small so the whole notebook runs fast on CPU.
H = 16                  # grid is H x H  (try 8 for a faster pass, 24 for a harder one)

# Reward modes: (center_x, center_y, height, width). Heights are the *bump*
# magnitude on top of a small background; widths control mode sharpness.
REWARD_MODES = [
    (1,  1,  2.0, 0.8),   # bottom-left
    (H-2, H-2, 2.0, 0.8), # top-right
    (H-2, 1, 1.5, 0.8),   # bottom-right
    (1, H-2, 1.5, 0.8),   # top-left
    (H//2, H//2, 1.0, 0.8)# center (a smaller mode)
]
R_BACKGROUND = 1e-2     # small positive background so R(x) > 0 everywhere

# Budgets (each method gets a comparable number of *generated terminal states*).
GFN_UPDATES   = 3000    # trajectory-balance gradient steps
GFN_BATCH     = 16      # trajectories per update
PG_UPDATES    = 3000    # REINFORCE/PPO updates
PG_BATCH      = 16
MCMC_STEPS    = GFN_UPDATES * GFN_BATCH   # match total samples for fairness
PG_ENTROPY    = 0.0     # entropy bonus for the policy-gradient baseline (0 => pure reward-max)

LR            = 1e-2
HIDDEN        = 64
SEEDS         = [0, 1, 2]   # repeat the mode-discovery curves over a few seeds

# How we count a "mode as discovered": a generated sample lands within this
# Chebyshev radius of a mode center.
MODE_RADIUS   = 1

ELAPSED = {}  # method -> seconds, filled in during training

def set_seed(s):
    random.seed(s); np.random.seed(s)
    if HAVE_TORCH:
        torch.manual_seed(s)

set_seed(SEEDS[0])
print(f"Grid {H}x{H} = {H*H} terminal cells | {len(REWARD_MODES)} reward modes")


## 2. Environment & reward

### 2.1 Reward landscape

$R(x,y) = \text{background} + \sum_m h_m \exp\!\big(-\tfrac{(x-c^x_m)^2+(y-c^y_m)^2}{2(\text{width}_m\cdot s)^2}\big)$

with $s$ a small scale so the bumps are *sharp and separated* (multi-modal). The
target distribution is just $R$ normalized to sum to 1.

In [ ]:
def build_reward(H, modes, background, scale=1.0):
    '''Return an (H,H) numpy array R[x,y] > 0 with several separated bumps.'''
    xs = np.arange(H)[:, None]   # (H,1) -> x index
    ys = np.arange(H)[None, :]   # (1,H) -> y index
    R = np.full((H, H), background, dtype=np.float64)
    for (cx, cy, h, w) in modes:
        sig2 = 2.0 * (w * scale) ** 2
        R = R + h * np.exp(-((xs - cx) ** 2 + (ys - cy) ** 2) / sig2)
    return R

R_GRID = build_reward(H, REWARD_MODES, R_BACKGROUND)
Z_TRUE = R_GRID.sum()
P_TRUE = R_GRID / Z_TRUE                       # target distribution p*(x) ∝ R(x)

# Mode "centers" we will try to recover, as (x,y) integer cells.
MODE_CENTERS = [(cx, cy) for (cx, cy, h, w) in REWARD_MODES]

# Sanity checks on the reward.
assert R_GRID.shape == (H, H)
assert (R_GRID > 0).all(), "reward must be strictly positive everywhere"
# Each declared mode center should be a strong local value (>= background+something).
for (cx, cy) in MODE_CENTERS:
    assert R_GRID[cx, cy] > 5 * R_BACKGROUND
print("Z_TRUE =", round(Z_TRUE, 4), "| max R =", round(R_GRID.max(), 4),
      "| min R =", round(R_GRID.min(), 4))


In [ ]:
# Quick look at the target landscape.
fig, ax = plt.subplots(1, 1, figsize=(4.2, 3.6))
im = ax.imshow(P_TRUE.T, origin="lower", cmap="viridis")
ax.set_title("Target  p*(x) ∝ R(x)")
ax.set_xlabel("x"); ax.set_ylabel("y")
for (cx, cy) in MODE_CENTERS:
    ax.scatter([cx], [cy], c="red", s=18, marker="x")
fig.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout(); plt.show()


### 2.2 The generative MDP (the DAG)

State = current cell $(x,y)$. Actions:

- `0` = `+x` (only legal if `x < H-1`),
- `1` = `+y` (only legal if `y < H-1`),
- `2` = `stop` (always legal; terminates).

Because you can reach a cell by many orderings of `+x`/`+y` moves, **every cell has
multiple parents** — the state graph is a DAG, not a tree. This multiplicity is
the whole point: the GFlowNet must account for *all* trajectories into a state,
which is what the trajectory-balance objective does and what plain RL ignores.

We encode a state as a one-hot over $H\times H$ cells (cheap, exact) so the policy
nets are tiny MLPs.

In [ ]:
STOP = 2  # action id for terminate
N_ACT = 3 # +x, +y, stop

def legal_mask(x, y, H):
    '''Boolean mask over [ +x, +y, stop ] for a single (x,y).'''
    m = np.array([x < H - 1, y < H - 1, True], dtype=bool)
    return m

def step(x, y, a, H):
    '''Apply action a; return (nx, ny, done). 'stop' keeps the cell, sets done.'''
    if a == 0:   return x + 1, y, False
    if a == 1:   return x, y + 1, False
    return x, y, True  # stop

def n_parents(x, y):
    '''Number of parents of cell (x,y) in the DAG = number of incoming non-stop
    edges. (0,0) has the single initial 'source' as parent.'''
    return max(1, (x > 0) + (y > 0))

def state_index(x, y, H):
    return x * H + y

def onehot(x, y, H):
    v = np.zeros(H * H, dtype=np.float32)
    v[state_index(x, y, H)] = 1.0
    return v

# --- env asserts ---
assert list(legal_mask(0, 0, H)) == [True, True, True]
assert list(legal_mask(H-1, H-1, H)) == [False, False, True]   # only stop at far corner
assert step(0, 0, 0, H) == (1, 0, False)
assert step(0, 0, 1, H) == (0, 1, False)
assert step(3, 4, STOP, H) == (3, 4, True)
assert n_parents(0, 0) == 1 and n_parents(2, 0) == 1 and n_parents(2, 3) == 2
print("env OK")


## 3. Method A — GFlowNet (Trajectory Balance)

A trajectory is $\tau = s_0 \to s_1 \to \dots \to s_n \to x$ where $s_0=(0,0)$, the
non-stop moves build up the cell, and the final `stop` emits terminal object $x$.

**Trajectory Balance (TB) loss** (Malkin et al., 2022) for one trajectory:

$$\mathcal{L}_{TB}(\tau)=\Big(\log Z + \sum_{t}\log P_F(a_t\mid s_t)\;-\;\log R(x)\;-\;\sum_{t}\log P_B(s_t\mid s_{t+1})\Big)^2$$

- $Z$ is a single learned scalar (we learn $\log Z$).
- $P_F(a\mid s)$ is the forward policy (softmax over **legal** actions).
- $P_B(s\mid s')$ is the backward policy; here **uniform over parents**, so for a
  forward non-stop move into a state $s'$ with $k$ parents, $\log P_B = -\log k$.
  The terminating `stop` edge contributes $\log P_B = 0$ (the terminal object has
  exactly one "parent" = the cell it sits on, via the stop edge).

At the global optimum, TB forces the forward policy to generate terminal states
**with probability $\propto R(x)$** — i.e. it samples the full multi-modal target,
not its argmax. That is the property we will verify empirically.

In [ ]:
class ForwardPolicy(nn.Module):
    '''Tiny MLP: state one-hot -> logits over [ +x, +y, stop ]. Also holds logZ.'''
    def __init__(self, H, hidden=HIDDEN):
        super().__init__()
        self.H = H
        self.net = nn.Sequential(
            nn.Linear(H * H, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, N_ACT),
        )
        # single global scalar logZ (the TB partition estimate)
        self.logZ = nn.Parameter(torch.zeros(1))

    def forward(self, onehots):
        return self.net(onehots)   # (B, N_ACT) raw logits

def masked_log_softmax(logits, mask):
    '''log-softmax over legal actions only (mask: bool tensor, same shape).'''
    neg = torch.finfo(logits.dtype).min
    logits = torch.where(mask, logits, torch.full_like(logits, neg))
    return torch.log_softmax(logits, dim=-1)


In [ ]:
def sample_trajectories_gfn(policy, batch, H, device="cpu", greedy=False):
    '''Roll out `batch` trajectories under the current forward policy.

    Returns:
      logPF_sum : (batch,) sum of log P_F(a_t|s_t) along each trajectory
      logPB_sum : (batch,) sum of log P_B(s_t|s_{t+1}) (uniform over parents)
      term_xy   : list of (x,y) terminal cells
    '''
    logPF_sum = torch.zeros(batch, device=device)
    logPB_sum = torch.zeros(batch, device=device)
    xs = [0] * batch; ys = [0] * batch
    done = [False] * batch
    max_steps = 2 * H  # an upper bound on trajectory length (then force stop)

    for t in range(max_steps + 1):
        active = [i for i in range(batch) if not done[i]]
        if not active:
            break
        force_stop = (t == max_steps)  # safety: terminate any stragglers
        oh = torch.tensor(np.stack([onehot(xs[i], ys[i], H) for i in active]),
                          device=device)
        masks = torch.tensor(np.stack([legal_mask(xs[i], ys[i], H) for i in active]))
        logits = policy(oh)
        logp = masked_log_softmax(logits, masks)            # (A, N_ACT)
        if force_stop:
            acts = torch.full((len(active),), STOP, dtype=torch.long)
        elif greedy:
            acts = logp.argmax(dim=-1)
        else:
            acts = torch.distributions.Categorical(logits=logp).sample()
        # accumulate logP_F for chosen actions
        chosen_logpf = logp.gather(1, acts.view(-1, 1)).squeeze(1)
        for j, i in enumerate(active):
            a = int(acts[j].item())
            logPF_sum[i] = logPF_sum[i] + chosen_logpf[j]
            if a == STOP:
                done[i] = True
                # terminating edge: P_B over the single 'stop parent' = 1 -> +0
            else:
                nx, ny, _ = step(xs[i], ys[i], a, H)
                # backward prob of having come from (xs,ys): uniform over parents
                # of the *child* state (nx,ny).
                logPB_sum[i] = logPB_sum[i] - math.log(n_parents(nx, ny))
                xs[i], ys[i] = nx, ny
    term_xy = list(zip(xs, ys))
    return logPF_sum, logPB_sum, term_xy


In [ ]:
def train_gflownet(H, R_grid, updates=GFN_UPDATES, batch=GFN_BATCH,
                   lr=LR, seed=0, track_tv_every=100, verbose=True):
    '''Train the forward policy with the Trajectory-Balance loss.

    Returns dict with: policy, loss_hist, tv_hist (TV vs P_TRUE at checkpoints),
    tv_steps, samples (terminal cells generated *during* training).
    '''
    set_seed(seed)
    device = "cpu"
    policy = ForwardPolicy(H).to(device)
    opt = torch.optim.Adam(policy.parameters(), lr=lr)
    logR = torch.tensor(np.log(R_grid), dtype=torch.float32)  # (H,H)

    loss_hist, tv_hist, tv_steps = [], [], []
    seen_counts = np.zeros((H, H), dtype=np.float64)  # empirical visit counts
    samples_in_order = []  # terminal cells, for mode-discovery curve

    t0 = time.time()
    for it in range(updates):
        logPF, logPB, term_xy = sample_trajectories_gfn(policy, batch, H, device)
        logRx = torch.stack([logR[x, y] for (x, y) in term_xy])
        # TB residual: logZ + sum logPF - logR - sum logPB
        resid = policy.logZ + logPF - logRx - logPB
        loss = (resid ** 2).mean()
        opt.zero_grad(); loss.backward(); opt.step()

        loss_hist.append(loss.item())
        for (x, y) in term_xy:
            seen_counts[x, y] += 1
            samples_in_order.append((x, y))

        if (it % track_tv_every == 0) or (it == updates - 1):
            emp = empirical_dist_gfn(policy, H, n=2000)
            tv = 0.5 * np.abs(emp - P_TRUE).sum()
            tv_hist.append(tv); tv_steps.append(it)
            if verbose and (it % (track_tv_every * 5) == 0):
                print(f"  [GFN] it={it:4d}  loss={loss.item():.3f}  "
                      f"logZ={policy.logZ.item():.3f}  TV={tv:.3f}")
    el = time.time() - t0
    ELAPSED["GFlowNet"] = ELAPSED.get("GFlowNet", 0.0) + el
    if verbose:
        print(f"  [GFN] done in {el:.1f}s | final logZ={policy.logZ.item():.3f} "
              f"(log Z_true={math.log(Z_TRUE):.3f})")
    return dict(policy=policy, loss_hist=loss_hist, tv_hist=tv_hist,
                tv_steps=tv_steps, seen_counts=seen_counts,
                samples=samples_in_order)

def empirical_dist_gfn(policy, H, n=4000):
    '''Sample n terminal states on-policy; return normalized (H,H) histogram.'''
    with torch.no_grad():
        _, _, term = sample_trajectories_gfn(policy, n, H)
    hist = np.zeros((H, H), dtype=np.float64)
    for (x, y) in term:
        hist[x, y] += 1
    return hist / hist.sum()


## 4. Method B — REINFORCE / PPO (reward maximization)

Same forward-policy network and same MDP, but now we **maximize expected reward**
with a policy gradient (REINFORCE with a moving-average baseline; an optional
entropy bonus). This is the FlowGRPO / GRPO-style objective stripped to its
essence:

$$\nabla_\theta\,\mathbb{E}_{\tau\sim P_F}[R(x)]
 \;=\; \mathbb{E}\big[(R(x)-b)\,\nabla_\theta \textstyle\sum_t \log P_F(a_t\mid s_t)\big].$$

There is **no** $\log Z$ and **no** backward policy — the optimum is the policy that
puts *all* its mass on the single highest-reward terminal cell. So we expect
**mode collapse**: the empirical distribution becomes a spike, even though the
reward has several modes. (Entropy bonus softens but does not fix this; it does
not make the policy sample $\propto R$.)

In [ ]:
def train_reinforce(H, R_grid, updates=PG_UPDATES, batch=PG_BATCH, lr=LR,
                    seed=0, entropy_coef=PG_ENTROPY, verbose=True):
    '''REINFORCE with a baseline (a minimal PPO-free policy-gradient baseline).

    Returns dict with policy, reward_hist, seen_counts, samples.
    '''
    set_seed(seed)
    device = "cpu"
    policy = ForwardPolicy(H).to(device)
    opt = torch.optim.Adam(policy.parameters(), lr=lr)
    Rt = torch.tensor(R_grid, dtype=torch.float32)
    baseline = 0.0
    reward_hist = []
    seen_counts = np.zeros((H, H), dtype=np.float64)
    samples_in_order = []

    t0 = time.time()
    for it in range(updates):
        # roll out, collecting per-trajectory logPF and entropy
        logPF = torch.zeros(batch, device=device)
        ent   = torch.zeros(batch, device=device)
        xs = [0]*batch; ys=[0]*batch; done=[False]*batch
        max_steps = 2*H
        for t in range(max_steps+1):
            active=[i for i in range(batch) if not done[i]]
            if not active: break
            force_stop = (t==max_steps)
            oh = torch.tensor(np.stack([onehot(xs[i],ys[i],H) for i in active]))
            masks = torch.tensor(np.stack([legal_mask(xs[i],ys[i],H) for i in active]))
            logp = masked_log_softmax(policy(oh), masks)
            p = logp.exp()
            if force_stop:
                acts = torch.full((len(active),), STOP, dtype=torch.long)
            else:
                acts = torch.distributions.Categorical(logits=logp).sample()
            chosen = logp.gather(1, acts.view(-1,1)).squeeze(1)
            step_ent = -(p * logp).sum(dim=-1)
            for j,i in enumerate(active):
                logPF[i] = logPF[i] + chosen[j]
                ent[i]   = ent[i] + step_ent[j]
                a=int(acts[j].item())
                if a==STOP: done[i]=True
                else: xs[i],ys[i],_=step(xs[i],ys[i],a,H)
        rew = torch.stack([Rt[x,y] for (x,y) in zip(xs,ys)])
        baseline = 0.95*baseline + 0.05*rew.mean().item()
        adv = (rew - baseline).detach()
        loss = -((adv * logPF).mean()) - entropy_coef * ent.mean()
        opt.zero_grad(); loss.backward(); opt.step()

        reward_hist.append(rew.mean().item())
        for (x,y) in zip(xs,ys):
            seen_counts[x,y]+=1; samples_in_order.append((x,y))
        if verbose and (it % (max(1,updates//6))==0):
            print(f"  [PG ] it={it:4d}  meanR={rew.mean().item():.3f}  "
                  f"baseline={baseline:.3f}")
    el = time.time()-t0
    ELAPSED["REINFORCE"] = ELAPSED.get("REINFORCE",0.0)+el
    if verbose: print(f"  [PG ] done in {el:.1f}s")
    return dict(policy=policy, reward_hist=reward_hist,
                seen_counts=seen_counts, samples=samples_in_order)

def empirical_dist_pg(policy, H, n=4000):
    '''On-policy empirical terminal distribution for the PG policy.'''
    with torch.no_grad():
        _,_,term = sample_trajectories_gfn(policy, n, H)  # same rollout works
    hist=np.zeros((H,H))
    for (x,y) in term: hist[x,y]+=1
    return hist/hist.sum()


## 5. Method C — MCMC (Metropolis over terminal cells)

A classic baseline that **is** asymptotically $\propto R$, so it is the "right"
target — but it explores by *local* moves and therefore **mixes slowly** between
separated modes. We run a Metropolis random walk directly on terminal cells
$(x,y)$ with proposal = step to a random 4-neighbour, accept with probability
$\min(1, R(x')/R(x))$. Because the modes are separated by a low-reward background,
the chain gets stuck in whatever basin it starts in for long stretches.

In [ ]:
def run_mcmc(H, R_grid, steps=MCMC_STEPS, seed=0, start=(0,0)):
    '''Metropolis random walk on the grid targeting p ∝ R. Returns visit
    counts, the ordered list of visited cells, and the (normalized) empirical
    distribution.'''
    set_seed(seed)
    x, y = start
    counts = np.zeros((H, H), dtype=np.float64)
    visited = []
    nbrs = [(1,0),(-1,0),(0,1),(0,-1)]
    t0 = time.time()
    for _ in range(steps):
        dx, dy = nbrs[np.random.randint(4)]
        nx, ny = x+dx, y+dy
        if 0 <= nx < H and 0 <= ny < H:
            if np.random.rand() < min(1.0, R_grid[nx,ny]/R_grid[x,y]):
                x, y = nx, ny
        counts[x,y]+=1; visited.append((x,y))
    el = time.time()-t0
    ELAPSED["MCMC"] = ELAPSED.get("MCMC",0.0)+el
    emp = counts/counts.sum()
    return dict(counts=counts, visited=visited, emp=emp)


## 6. Diversity metric: #modes discovered vs sampling budget

We say a mode is **discovered** once *any* generated sample lands within Chebyshev
radius `MODE_RADIUS` of its center. We sweep the ordered stream of samples each
method produced and record the cumulative count of distinct modes hit — averaged
over seeds. GFlowNet should climb to *all* modes; REINFORCE should plateau at ~1.

In [ ]:
def modes_discovered_curve(samples, mode_centers, radius=MODE_RADIUS, n_points=60):
    '''Given an ordered list of (x,y) samples, return (budgets, n_modes) where
    n_modes[k] = #distinct modes hit within the first budgets[k] samples.'''
    N = len(samples)
    budgets = np.unique(np.linspace(1, N, n_points).astype(int))
    found = set()
    curve = []
    ptr = 0
    for b in budgets:
        while ptr < b:
            sx, sy = samples[ptr]
            for mi, (cx, cy) in enumerate(mode_centers):
                if max(abs(sx-cx), abs(sy-cy)) <= radius:
                    found.add(mi)
            ptr += 1
        curve.append(len(found))
    return budgets, np.array(curve)

def avg_curve_over_seeds(sample_lists, mode_centers):
    '''Interpolate per-seed curves onto a common budget grid and average.'''
    common = None; stacked = []
    for s in sample_lists:
        b, c = modes_discovered_curve(s, mode_centers)
        if common is None:
            common = np.linspace(1, min(len(x) for x in sample_lists), 60).astype(int)
        stacked.append(np.interp(common, b, c))
    return common, np.mean(stacked, axis=0), np.std(stacked, axis=0)


## 7. Train all three methods (across seeds)

This is the main compute. With the defaults (H=16, 3000 updates, 3 seeds) it
runs in roughly a minute or two on a Colab CPU.

In [ ]:
gfn_runs, pg_runs, mcmc_runs = [], [], []
T0 = time.time()
for si, sd in enumerate(SEEDS):
    print(f"=== seed {sd} ({si+1}/{len(SEEDS)}) ===")
    print(" GFlowNet ...")
    gfn_runs.append(train_gflownet(H, R_GRID, seed=sd, verbose=(si==0)))
    print(" REINFORCE/PPO ...")
    pg_runs.append(train_reinforce(H, R_GRID, seed=sd, verbose=(si==0)))
    print(" MCMC ...")
    mcmc_runs.append(run_mcmc(H, R_GRID, seed=sd))
print(f"\\nTOTAL wall time: {time.time()-T0:.1f}s")
print("per-method elapsed (s):", {k: round(v,1) for k,v in ELAPSED.items()})


## 8. Results

### 8.1 Heatmaps — empirical terminal distribution vs the true target

GFlowNet's histogram should **look like** $p^\*(x)$ (all modes lit up).
REINFORCE's should be a **single spike**. MCMC should light up modes but with
uneven, seed-dependent mass (slow mixing).

In [ ]:
# Use seed-0 trained models for the heatmaps; sample fresh empirical dists.
gfn_emp = empirical_dist_gfn(gfn_runs[0]["policy"], H, n=8000)
pg_emp  = empirical_dist_pg(pg_runs[0]["policy"],  H, n=8000)
mcmc_emp= mcmc_runs[0]["emp"]

panels = [("True  p* ∝ R", P_TRUE),
          ("GFlowNet (∝ R)", gfn_emp),
          ("REINFORCE/PPO (argmax)", pg_emp),
          ("MCMC (slow mixing)", mcmc_emp)]
fig, axes = plt.subplots(1, 4, figsize=(15, 3.6))
for ax, (title, M) in zip(axes, panels):
    im = ax.imshow(M.T, origin="lower", cmap="viridis")
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("x"); ax.set_ylabel("y")
    for (cx,cy) in MODE_CENTERS:
        ax.scatter([cx],[cy], c="red", s=10, marker="x")
    fig.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout(); plt.show()

# Quantitative agreement with the target.
def tv(p,q): return 0.5*np.abs(p-q).sum()
def pearson(p,q):
    a,b = p.ravel(), q.ravel()
    return float(np.corrcoef(a,b)[0,1])
print(f"TV(empirical, p*)   GFN={tv(gfn_emp,P_TRUE):.3f}  "
      f"PG={tv(pg_emp,P_TRUE):.3f}  MCMC={tv(mcmc_emp,P_TRUE):.3f}")
print(f"Pearson r vs p*     GFN={pearson(gfn_emp,P_TRUE):.3f}  "
      f"PG={pearson(pg_emp,P_TRUE):.3f}  MCMC={pearson(mcmc_emp,P_TRUE):.3f}")


### 8.2 Diversity — number of distinct modes discovered vs sampling budget

The headline plot: GFlowNet finds **all** the modes; REINFORCE/PPO saturates near
**one**; MCMC is in between and noisier (depends where the chain happens to wander).

In [ ]:
gfn_samples  = [r["samples"] for r in gfn_runs]
pg_samples   = [r["samples"] for r in pg_runs]
mcmc_samples = [r["visited"] for r in mcmc_runs]

b_g, m_g, s_g = avg_curve_over_seeds(gfn_samples,  MODE_CENTERS)
b_p, m_p, s_p = avg_curve_over_seeds(pg_samples,   MODE_CENTERS)
b_m, m_m, s_m = avg_curve_over_seeds(mcmc_samples, MODE_CENTERS)

plt.figure(figsize=(6.5,4.2))
for (b,m,s,lbl,c) in [(b_g,m_g,s_g,"GFlowNet","C0"),
                      (b_p,m_p,s_p,"REINFORCE/PPO","C3"),
                      (b_m,m_m,s_m,"MCMC","C2")]:
    plt.plot(b,m,label=lbl,color=c)
    plt.fill_between(b, m-s, m+s, alpha=0.15, color=c)
plt.axhline(len(MODE_CENTERS), ls="--", c="gray", lw=1, label="all modes")
plt.xlabel("samples generated (budget)"); plt.ylabel("# distinct modes discovered")
plt.title(f"Mode discovery vs budget  (mean ± std over {len(SEEDS)} seeds)")
plt.legend(); plt.tight_layout(); plt.show()

print("final #modes  GFN=%.2f  PG=%.2f  MCMC=%.2f  (of %d)" %
      (m_g[-1], m_p[-1], m_m[-1], len(MODE_CENTERS)))


### 8.3 GFlowNet convergence — TB loss and TV-to-target over training

Trajectory-balance loss should fall, and the total-variation distance between the
GFlowNet's empirical distribution and $p^\*$ should drop toward 0 — i.e. it is
*learning to sample the whole landscape*, not to maximize.

In [ ]:
fig, (axL, axR) = plt.subplots(1,2, figsize=(11,3.8))
# TB loss (smoothed) for seed 0
loss = np.array(gfn_runs[0]["loss_hist"])
k = max(1, len(loss)//100)
sm = np.convolve(loss, np.ones(k)/k, mode="valid")
axL.plot(sm); axL.set_yscale("log")
axL.set_title("GFlowNet: Trajectory-Balance loss (smoothed)")
axL.set_xlabel("update"); axL.set_ylabel("TB loss (log)")

# TV to target over training, averaged over seeds
tv_steps = gfn_runs[0]["tv_steps"]
tv_stack = np.stack([np.array(r["tv_hist"]) for r in gfn_runs])
axR.plot(tv_steps, tv_stack.mean(0), color="C0")
axR.fill_between(tv_steps, tv_stack.mean(0)-tv_stack.std(0),
                 tv_stack.mean(0)+tv_stack.std(0), alpha=0.2, color="C0")
# reference: PG's TV (constant-ish, computed once)
pg_tv = tv(empirical_dist_pg(pg_runs[0]["policy"],H,4000), P_TRUE)
axR.axhline(pg_tv, ls="--", c="C3", label=f"REINFORCE TV ≈ {pg_tv:.2f}")
axR.set_title("GFlowNet: TV(empirical, p*) over training")
axR.set_xlabel("update"); axR.set_ylabel("total variation"); axR.legend()
plt.tight_layout(); plt.show()


## 9. Takeaways

1. **Sampling $\propto R$ vs $\arg\max R$ is the whole story.** GFlowNet's TB
   objective drives the forward policy to emit terminal objects with probability
   proportional to reward, so it *covers every mode* — visible as a heatmap that
   matches $p^\*$ and a mode-discovery curve that reaches **all** modes.
   REINFORCE/PPO optimizes $\mathbb{E}[R]$ and therefore **collapses to the single
   best cell**: a spike heatmap and a mode curve stuck near 1.

2. **MCMC is "correct but slow."** Metropolis targets $\propto R$ in the limit,
   but local moves across a low-reward background mean it mixes slowly between
   separated modes — its mode-discovery curve is noisier and seed-dependent, and
   its heatmap has uneven mass. GFlowNet *amortizes* this exploration into a
   neural sampler you can draw i.i.d. batches from.

3. **Why this matters for molecular / binder design.** A reward model is a noisy
   proxy; you want a **diverse batch** of distinct high-reward candidates, not one
   exploitable peak. This is the diversity-first argument for GFlowNets over
   reward-maximizing RL in my notes on
   **[GRPO](https://app.notion.com/p/37f95008d76681499281fcd797cb3d1d)** and
   **[Search + Learning for Molecular Design](https://app.notion.com/p/37f95008d7668176886cf77d703358bc)**
   (GFlowNet as the diversity-first rival to MCTS / RL for synthesis-tree
   generation). The hypergrid here is the minimal analogue of a synthesis DAG:
   many paths into each object, several separated reward basins.

### Knobs to explore
- Raise `H` and add more `REWARD_MODES` -> the RL/GFN gap widens.
- Set `PG_ENTROPY > 0` -> RL spreads a little but still does **not** sample $\propto R$.
- Sharpen modes (smaller `width`) or lower `R_BACKGROUND` -> MCMC mixing gets worse.
- Replace uniform $P_B$ with a learned backward policy (full TB) as an exercise.

*Elapsed times per method are printed above; the whole notebook is CPU-only.*
